# Collected 100-receipt external-test analysis
Read-only analysis. All model runs and attacks are submitted from the CARC terminal.

In [ ]:
from pathlib import Path
import sys, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
from result_analysis import load_table, rate_table, parse_paths, show_comparison
BASE = ROOT/'outputs/baseline/baseline_collected_external_v1/merged/records.parquet'
QWEN = ROOT/'outputs/qwen_attack/qwen_collected_external_v1/merged/records.parquet'
DONUT = ROOT/'outputs/donut_attack/donut_collected_external_v1/merged/records.parquet'


In [ ]:
baseline = load_table(BASE)
baseline.groupby('model_key').agg(documents=('record_id','size'), clean_correct=('clean_correct','sum'), clean_accuracy=('clean_correct','mean'), parse_failures=('parse_status', lambda x: (x!='ok').sum()))


In [ ]:
attacks = pd.concat([load_table(QWEN).assign(model_key='qwen'), load_table(DONUT).assign(model_key='donut')], ignore_index=True)
rate_table(attacks.query("status == 'complete'"), ['model_key','epsilon_255'], 'saved_image_success')


## High-resolution example
Change `MODEL`, `EPSILON`, and `SUCCESS`, then run the cell. It displays one document at a time at high DPI.

In [ ]:
MODEL, EPSILON, SUCCESS = 'qwen', 16, True
row = attacks.query('model_key == @MODEL and epsilon_255 == @EPSILON and saved_image_success == @SUCCESS').iloc[0]
manifest = pd.read_csv(ROOT/'data/processed/collected_receipts/manifest.csv').set_index('record_id')
clean_path = ROOT/manifest.loc[row.record_id, 'processed_path']
attack_path = Path(parse_paths(row.images)[0] if 'images' in row and parse_paths(row.images) else row.image_path)
show_comparison(clean_path, attack_path, title=f'{row.record_id} | {MODEL} | ε={EPSILON} | success={SUCCESS}')
